# Kaggle runner — exp_0021 / exp_0022 (TabPFN baseline)

Trains the in-context family on GPU, on the **frozen 5-fold partition** shipped as the
`s6e7-frozen-folds` dataset (never rebuilt — rule 6):

- **exp_0021** — TabPFN-3, a class-proportional 100k-row context per fold,
  `balance_probabilities=True` (the zero-parameter 1/π rule inside `predict_proba`),
  13 raw features. One variable vs exp_0004: the model family.
- **exp_0022** — same + the 9 exact-value target-encoding columns of exp_0017
  (`encoders.exact_value_te`, fitted on the fold's full 552k fit rows). One variable vs
  exp_0021.

**Requires a Kaggle secret named `TABPFN_TOKEN`** (Add-ons → Secrets): the weights are
gated behind a free Prior Labs account and a non-commercial licence — accept it at
https://ux.priorlabs.ai (License tab), copy the token from the account page. Without the
secret the token cell fails loudly, before any GPU time is spent.

**Carry back** (`kaggle kernels output`): `artifacts/exp_00{21,22}{,_test}.npy` →
`oof/`, and the two new rows of `artifacts/experiments.csv` → the local ledger.

In [ ]:
# Diagnostics first, stdlib only. Kaggle's log endpoint has returned empty files for
# errored runs, so this cell makes the run self-reporting: mounts go to an output file,
# and any later cell's exception is written to diag_error.txt before it propagates
# (output files survive an errored run).
import json
import sys
import traceback
from pathlib import Path

diag = {
    "python": sys.version,
    "inputs": {
        p.name: sorted(f.name for f in p.iterdir())[:10]
        for p in Path("/kaggle/input").iterdir()
    },
}
Path("/kaggle/working/diag_mounts.json").write_text(json.dumps(diag, indent=2))
print(json.dumps(diag, indent=2))


def _dump_exc(shell, etype, evalue, tb, tb_offset=None):
    text = "".join(traceback.format_exception(etype, evalue, tb))
    Path("/kaggle/working/diag_error.txt").write_text(text)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


get_ipython().set_custom_exc((BaseException,), _dump_exc)

In [ ]:
import torch

torch_before = torch.__version__
%pip install -q tabpfn
import importlib

importlib.reload(torch)
# tabpfn pins torch>=2.5 only, so pip should leave the image's CUDA torch alone; if it
# did not, stop here rather than train on a wheel without kernels for this GPU.
assert torch.__version__ == torch_before, f"pip replaced torch: {torch_before} -> {torch.__version__}"
assert torch.cuda.is_available(), "torch lost CUDA during install"
print("torch", torch.__version__, "cuda ok")

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
os.environ["TABPFN_NO_BROWSER"] = "1"          # headless: never try to open a login page
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/tmp/tabpfn_cache"  # keep weights out of the output
assert os.environ["TABPFN_TOKEN"], "TABPFN_TOKEN secret is empty"
print("token present:", len(os.environ["TABPFN_TOKEN"]), "chars")

In [ ]:
import shutil
import subprocess

# The repo lives in /tmp, NOT /kaggle/working: only artifacts/ and diag files should
# land in the output snapshot, so a failed run stays kilobytes and pulls in seconds.
REPO = Path("/tmp/repo")

clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/epsilonlog/comp-playground-series-s6e7.git", str(REPO)],
    capture_output=True, text=True,
)
assert (REPO / "src").exists(), f"clone failed: {clone.stderr}"

inputs = Path("/kaggle/input")


def first_existing(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"none exist: {[str(c) for c in candidates]}")


# Mount layout differs across Kaggle images: classic /kaggle/input/<slug> vs the
# nested /kaggle/input/competitions/<slug> and /kaggle/input/datasets/<owner>/<slug>.
comp = first_existing(
    inputs / "competitions" / "playground-series-s6e7",
    inputs / "playground-series-s6e7",
)
ds = first_existing(
    inputs / "datasets" / "aligh474" / "s6e7-frozen-folds",
    inputs / "s6e7-frozen-folds",
)

raw = REPO / "data" / "raw"
processed = REPO / "data" / "processed"
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

for n in ["train.csv", "test.csv", "sample_submission.csv"]:
    src = comp / n if (comp / n).exists() else ds / n
    assert src.exists(), f"{n} found in neither {comp} nor {ds}"
    shutil.copy(src, raw / n)
shutil.copy(ds / "folds.parquet", processed / "folds.parquet")

print("csv source:", comp)
print("raw:", sorted(p.name for p in raw.iterdir()))
print("processed:", sorted(p.name for p in processed.iterdir()))

In [ ]:
import sys

sys.path.insert(0, "/tmp/repo/src")

import numpy as np
import tabpfn

from s6e7 import cv, folds, io
from s6e7.cv import ExperimentConfig

print("tabpfn", tabpfn.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
train, test = io.load_train(), io.load_test()
folds.verify(train)  # the shipped file IS the frozen partition; prove it arrived intact
print("folds verified:", folds.FOLDS_PATH)

Smoke first: weights download, licence/token accepted, one small context fits and
predicts on GPU, with and without the encoder. 34k rows, 5k context, never logged. The
scores are meaningless **by design**; the *timing* is the number to read — inference
cost scales with context × query rows, and it decides whether `max_fit_rows` can be
raised in a follow-up config.

In [ ]:
import time

smoke = train.head(34_000)
for encoder in ("", "exact_value_te"):
    t0 = time.perf_counter()
    r = cv.run(
        ExperimentConfig(exp_id="smoke_tabpfn", model="tabpfn",
                         params={"max_fit_rows": 5_000}, encoder=encoder),
        train=smoke,
        log=False,
    )
    label = "tabpfn + " + encoder if encoder else "tabpfn"
    print(f"{label}: plumbing ok, {time.perf_counter() - t0:.0f}s for 5 folds of "
          f"5k context x ~34k query rows (scores meaningless at this n)")

Full runs at the default 100k context. `cv.run` scores every fold model on its own
552k fit rows too (the fit–val gap), so each fold answers ~1M query rows; budget
accordingly and read the runtime before raising `max_fit_rows` in a follow-up config.

In [ ]:
result_21 = cv.run(
    ExperimentConfig(
        exp_id="exp_0021",
        model="tabpfn",
        parent="exp_0004",
        changed="new family: TabPFN-3 in-context (100k stratified context per fold, prior-balanced), raw features",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0021  cv_mean={result_21.cv_mean:.5f}  cv_std={result_21.cv_std:.5f}  runtime={result_21.runtime_s:.0f}s")
print("fold scores:", [round(s, 5) for s in result_21.fold_scores])
print("fit scores: ", [round(s, 5) for s in result_21.fit_scores], "(fit - val gap = the over/underfitting dial)")

In [ ]:
result_22 = cv.run(
    ExperimentConfig(
        exp_id="exp_0022",
        model="tabpfn",
        encoder="exact_value_te",
        parent="exp_0021",
        changed="add 9 fold-fitted exact-value TE columns (exp_0017's encoder)",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0022  cv_mean={result_22.cv_mean:.5f}  cv_std={result_22.cv_std:.5f}  runtime={result_22.runtime_s:.0f}s")
print("fold scores:", [round(s, 5) for s in result_22.fold_scores])
print("fit scores: ", [round(s, 5) for s in result_22.fit_scores], "(fit - val gap = the over/underfitting dial)")

In [ ]:
art = Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for rel in ["oof/exp_0021.npy", "oof/exp_0021_test.npy", "oof/exp_0022.npy", "oof/exp_0022_test.npy", "experiments.csv"]:
    shutil.copy(REPO / rel, art / Path(rel).name)
print(sorted(f"{p.name} ({p.stat().st_size:,}B)" for p in art.iterdir()))